# Silver_Merge_Execution

Runs after the quality gate passes. Merges the valid, staged order rows into `silver_orders` — the durable Silver table that Gold and reporting read from.

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from notebookutils import mssparkutils

silver_table_name = "silver_orders"

# orders_pending_routing is written by Orders_Bronze_to_Silver in the same
# pipeline run — it holds every row of the current file, tagged IsValid.
# Only the valid rows are kept; IsValid/FailureReason are routing metadata,
# not part of the Silver schema, so they're dropped here.
if not spark.catalog.tableExists("orders_pending_routing"):
    raise ValueError(
        "Table 'orders_pending_routing' was not found. "
        "Run Orders_Bronze_to_Silver first."
    )

df_incoming = (
    spark.read.table("orders_pending_routing")
    .filter(F.col("IsValid"))
    .drop("IsValid", "FailureReason")
)

In [ ]:
# Idempotent upsert: first run creates the table (partitioned by
# DeliveryRegion), later runs merge on OrderId — update matches, insert new rows.
if spark.catalog.tableExists(silver_table_name):
    delta_table = DeltaTable.forName(spark, silver_table_name)
    (
        delta_table.alias("target")
        .merge(df_incoming.alias("source"), "target.OrderId = source.OrderId")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print(f"Delta merge completed successfully into '{silver_table_name}'.")
else:
    df_incoming.write.format("delta").mode("overwrite") \
        .partitionBy("DeliveryRegion").saveAsTable(silver_table_name)
    print(f"Silver table '{silver_table_name}' initialized (partitioned by DeliveryRegion).")

# Signal completion to the pipeline.
mssparkutils.notebook.exit('{"status": "silver_merge_success"}')